# Testing ways to connect phrases using grammatical information.

In [1]:

import spacy
import networkx as nx
import matplotlib.pyplot as plt
from spacy_experimental.coref.coref_util import DEFAULT_COREF_MODEL

# Ensure you have run: python -m spacy download en_core_web_lg

def analyze_and_visualize_text(text: str):
    """
    Performs an advanced linguistic analysis on text and visualizes it as a graph.

    This function processes text to identify tokens, entities, syntactic dependencies,
    coreferences, and discourse relations. It then constructs and displays a
    networkx graph representing this information.

    Args:
        text: The input string to be analyzed.
    """
    print("🧠 Starting analysis...")

    # 1. SETUP & DEPENDENCIES
    # ========================
    # Load the large English spaCy model
    nlp = spacy.load("en_core_web_lg")

    # Add the experimental coreference component to the pipeline
    # The default config is loaded from spacy-experimental
    nlp.add_pipe("experimental_coref", config=DEFAULT_COREF_MODEL)

    # Define the discourse lexicon and create a reverse map for easy lookups
    DISCOURSE_LEXICON = {
        'contrast': ['however', 'but', 'whereas', 'despite'],
        'causation': ['consequently', 'therefore', 'because'],
        'elaboration': ['specifically', 'in other words'],
        'summary': ['ultimately', 'in conclusion']
    }
    REVERSE_DISCOURSE_LEXICON = {word: key for key, words in DISCOURSE_LEXICON.items() for word in words}

    # Process the text with the full pipeline
    doc = nlp(text)

    # Initialize a directed graph
    G = nx.DiGraph()

    # 2. NODE & EDGE DEFINITION (Initial Pass)
    # ========================================
    print("📊 Building graph structure...")

    # Add a node for each token
    for token in doc:
        G.add_node(token.i, text=token.text, pos=token.pos_)

    # Add syntactic dependency edges
    for token in doc:
        if token.head.i != token.i:
            G.add_edge(token.i, token.head.i, type='dependency', label=token.dep_)

    # 3. ADVANCED ANNOTATION (Sentence Level)
    # =======================================
    prev_sent_root = None
    for sent in doc.sents:
        # --- Negation & Modality Annotation ---
        for token in sent:
            # Check for negation and mark the root of the negated clause
            if token.dep_ == 'neg':
                G.nodes[token.head.i]['is_negated'] = True
            # Check for modal verbs and mark the root of the modalized clause
            if token.tag_ == 'MD':
                G.nodes[token.head.i]['modality'] = token.text

        # --- Discourse Mediation Annotation & Edges ---
        for token in sent:
            discourse_type = REVERSE_DISCOURSE_LEXICON.get(token.text.lower())
            if discourse_type:
                # Add attribute to the discourse mediator token node
                G.nodes[token.i]['discourse_type'] = discourse_type

                # Add directed discourse edge to the current sentence's root
                G.add_edge(token.i, sent.root.i, type='discourse')

                # Add directed discourse edge to the previous sentence's root
                if prev_sent_root:
                    G.add_edge(token.i, prev_sent_root.i, type='discourse')

        prev_sent_root = sent.root

    # 4. COREFERENCE EDGES
    # ====================
    if doc._.coref_clusters:
        for cluster in doc._.coref_clusters:
            # Get the root tokens of all mentions in the cluster
            mention_roots = [mention.root.i for mention in cluster.mentions]

            # Create edges between all pairs of mention roots in the cluster
            for i in range(len(mention_roots)):
                for j in range(i + 1, len(mention_roots)):
                    G.add_edge(mention_roots[i], mention_roots[j], type='coreference')

    # 5. GRAPH VISUALIZATION
    # ======================
    print("🎨 Generating visualization...")
    plt.figure(figsize=(22, 16), dpi=100)

    # Use Kamada-Kawai layout for a visually appealing structure
    pos = nx.kamada_kawai_layout(G)

    # --- Node Styling ---
    node_labels = nx.get_node_attributes(G, 'text')
    node_colors = []
    pos_color_map = {
        'NOUN': '#b2df8a', 'PROPN': '#b2df8a', 'PRON': '#b2df8a', # Nouns
        'VERB': '#fdbf6f', 'AUX': '#fdbf6f', # Verbs
        'ADJ': '#cab2d6', 'ADV': '#cab2d6', # Adverbs/Adjectives
        'ADP': '#a6cee3', 'SCONJ': '#a6cee3', # Conjunctions
        'PUNCT': '#e31a1c'
    }

    for node in G.nodes():
        if G.nodes[node].get('discourse_type'):
            node_colors.append('#ffff99') # Highlight discourse mediators
        else:
            node_colors.append(pos_color_map.get(G.nodes[node]['pos'], '#cccccc')) # Default color

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2500, edgecolors='black')

    # --- Edge Styling ---
    dep_edges = [(u, v) for u, v, d in G.edges(data=True) if d['type'] == 'dependency']
    coref_edges = [(u, v) for u, v, d in G.edges(data=True) if d['type'] == 'coreference']
    discourse_edges = [(u, v) for u, v, d in G.edges(data=True) if d['type'] == 'discourse']

    # Draw dependency edges (standard)
    nx.draw_networkx_edges(G, pos, edgelist=dep_edges, width=1.0, alpha=0.6, edge_color='grey')

    # Draw coreference edges (dashed, blue)
    nx.draw_networkx_edges(G, pos, edgelist=coref_edges, width=2.0, alpha=0.8, edge_color='blue', style='dashed')

    # Draw discourse edges (dotted, red)
    nx.draw_networkx_edges(G, pos, edgelist=discourse_edges, width=2.5, alpha=1.0, edge_color='red', style='dotted', connectionstyle='arc3,rad=0.1')

    # --- Label Styling ---
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=9, font_weight='bold')

    plt.title("Advanced Linguistic Network", fontsize=20)
    plt.axis('off')
    plt.tight_layout()
    print("✅ Done! Displaying graph.")
    plt.show()


if __name__ == '__main__':
    # The input text for analysis
    INPUT_TEXT = """The task of text classification (TC), also known as text categorization, is a foundational problem in Natural Language Processing (NLP). Early statistical methods, such as Naive Bayes, treated documents as simple bags of words. While these models offered computational efficiency, however, they fundamentally struggled with semantic nuance and complex syntax.

A paradigm shift occurred with the advent of deep learning, a field heavily influenced by researchers like Geoffrey Hinton. Consequently, this evolution led to sophisticated architectures like Google's BERT. Specifically, this transformer-based model revolutionized text representation by capturing context in a way previously unimaginable. Ultimately, the entire journey from simple Bayesian classifiers to these complex transformers underscores a central theme: the quest to understand language. This foundational challenge continues to drive the field forward."""

    # Run the analysis and visualization
    analyze_and_visualize_text(INPUT_TEXT)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject